In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data Preparation

This notebook validates, cleans, merges, and aggregates the
hourly bicycle traffic and weather datasets into a single
daily forecasting dataset.

The resulting dataset is written to
`data/processed/bike_weather_daily.csv` and serves as the
input for the exploratory data analysis and forecasting
phases of the project.

In [26]:
from src.data.cleaning import parse_datetime, interpolate_missing, aggregate_daily
from src.data.loaders import load_bike_counts, load_weather

### Load Raw Data

The raw bicycle traffic and weather datasets are loaded
from the `data/raw` directory. These datasets will be
validated and cleaned before being merged into a single
forecasting dataset.

In [27]:
#Load Bike DF
bike_df = load_bike_counts(
    "../data/raw/fremont_bridge.csv"
)

#Load Weather DF
weather_df = load_weather(
    "../data/raw/weather.csv"
)

### Validate Bicycle Traffic Dataset

This section verifies the structure and integrity of the
raw bicycle traffic data. The dataset is inspected for
correct data types, duplicate timestamps, missing values,
and temporal consistency before any cleaning or merging
takes place.

In [28]:
#Validate Bike Data Set
bike_df = parse_datetime(bike_df, "date")
bike_df.head()

,date,fremont_bridge,fremont_bridge_nb,fremont_bridge_sb
0,2026-05-31 23:00:00,33.0,9.0,24.0
1,2026-05-31 22:00:00,51.0,20.0,31.0
2,2026-05-31 21:00:00,64.0,30.0,34.0
3,2026-05-31 20:00:00,148.0,72.0,76.0
4,2026-05-31 19:00:00,184.0,85.0,99.0


In [29]:
bike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119747 entries, 0 to 119746
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date               119747 non-null  datetime64[us]
 1   fremont_bridge     119717 non-null  float64       
 2   fremont_bridge_nb  119717 non-null  float64       
 3   fremont_bridge_sb  119717 non-null  float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 3.7 MB


### Validate Weather Dataset

This section verifies the structure and integrity of the
hourly weather observations. The dataset is inspected for
correct data types, duplicate timestamps, missing values,
and temporal consistency before it is merged with the
bicycle traffic data.

In [30]:
#Validate Weather Dataset
weather_df = parse_datetime(
    weather_df,
    "time"
)

weather_df.head()

,time,temperature_2m,precipitation,wind_speed_10m
0,2012-10-02 00:00:00,20.9,0.0,5.7
1,2012-10-02 01:00:00,19.2,0.0,6.3
2,2012-10-02 02:00:00,17.6,0.0,7.1
3,2012-10-02 03:00:00,16.6,0.0,8.9
4,2012-10-02 04:00:00,15.5,0.0,13.8


### Clean Bicycle Traffic Data

The bicycle traffic dataset is cleaned to ensure a
consistent hourly time series. Datetime values are
standardized, observations are sorted chronologically,
and the dataset is prepared for merging with the weather
data.

In [31]:
#Check for duplicate timestamps
print(bike_df["date"].duplicated().sum())

#Check for duplicate timestamps
print(bike_df["date"].diff().value_counts().head(10))

0
date
-1 days +23:00:00    119746
Name: count, dtype: int64


In [32]:
#Sort DataFrame
bike_df = bike_df.sort_values(
    "date"
).reset_index(drop=True)

bike_df["date"].diff().value_counts().head(10)

date
0 days 01:00:00    119746
Name: count, dtype: int64

In [33]:
#check duplicates
print(weather_df["time"].duplicated().sum())
print(weather_df["time"].diff().value_counts().head())

0
time
0 days 01:00:00    119759
Name: count, dtype: int64


### Merge Bicycle Traffic and Weather Data

The cleaned bicycle traffic and weather datasets are
merged using their timestamp columns. Combining these
datasets produces a single time series containing both
traffic counts and environmental variables, which forms
the basis for subsequent forecasting analyses.

In [34]:
#Merge

#Compare ranges
print(bike_df["date"].min())
print(bike_df["date"].max())

print(weather_df["time"].min())
print(weather_df["time"].max())

#Merge operation
bike_weather_hourly = bike_df.merge(
    weather_df,
    left_on="date",
    right_on="time",
    how="left"
)

#assertion check
assert bike_weather_hourly.shape[0] == bike_df.shape[0]

2012-10-02 13:00:00
2026-05-31 23:00:00
2012-10-02 00:00:00
2026-05-31 23:00:00


In [35]:
#Validate
print(bike_weather_hourly.shape)
print(bike_weather_hourly.isna().sum())

(119747, 8)
date                  0
fremont_bridge       30
fremont_bridge_nb    30
fremont_bridge_sb    30
time                  0
temperature_2m        0
precipitation         0
wind_speed_10m        0
dtype: int64


In [36]:
#Inspect missing time stamps for cleaning
missing_rows = bike_weather_hourly[
    bike_weather_hourly["fremont_bridge"].isna()
]

missing_rows["date"]

3805     2013-03-10 02:00:00
6116     2013-06-14 09:00:00
6117     2013-06-14 10:00:00
12541    2014-03-09 02:00:00
21277    2015-03-08 02:00:00
22342    2015-04-21 11:00:00
22343    2015-04-21 12:00:00
22344    2015-04-21 13:00:00
30181    2016-03-13 02:00:00
38917    2017-03-12 02:00:00
47653    2018-03-11 02:00:00
56389    2019-03-10 02:00:00
65125    2020-03-08 02:00:00
74029    2021-03-14 02:00:00
82765    2022-03-13 02:00:00
91501    2023-03-12 02:00:00
92387    2023-04-18 00:00:00
92388    2023-04-18 01:00:00
92389    2023-04-18 02:00:00
92390    2023-04-18 03:00:00
92391    2023-04-18 04:00:00
92392    2023-04-18 05:00:00
92393    2023-04-18 06:00:00
92394    2023-04-18 07:00:00
92395    2023-04-18 08:00:00
92396    2023-04-18 09:00:00
92397    2023-04-18 10:00:00
100237   2024-03-10 02:00:00
108973   2025-03-09 02:00:00
117709   2026-03-08 02:00:00
Name: date, dtype: datetime64[us]

The dataset contains 30 missing bicycle count observations.

Fourteen missing observations occur during annual daylight
saving time transitions at 2:00 AM and do not represent
sensor failures.

The remaining missing observations occur in three sensor
outage periods:

- June 14, 2013 (2 hours)
- April 21, 2015 (3 hours)
- April 18, 2023 (11 hours)

Total missingness represents approximately 0.025% of the
dataset.

In [37]:
#Interpolate
bike_weather_hourly = interpolate_missing(
    bike_weather_hourly,
    [
        "fremont_bridge",
        "fremont_bridge_nb",
        "fremont_bridge_sb"
    ]
)

#assertion(check)
assert (
    bike_weather_hourly[
        ["fremont_bridge",
         "fremont_bridge_nb",
         "fremont_bridge_sb"]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

#check for missing values
bike_weather_hourly.isna().sum()

date                 0
fremont_bridge       0
fremont_bridge_nb    0
fremont_bridge_sb    0
time                 0
temperature_2m       0
precipitation        0
wind_speed_10m       0
dtype: int64

### Aggregate Hourly Data to Daily Data

The merged dataset is aggregated from hourly observations
to daily observations to match the forecasting objective.
Daily bicycle traffic is computed as the total daily rider
count, while weather variables are summarized using
appropriate daily statistics (e.g., mean temperature,
total precipitation, and mean wind speed).

In [38]:
#Data aggregation
daily_df = aggregate_daily(
    bike_weather_hourly
)
print(daily_df.shape)
daily_df.head()

(4990, 5)


,date,fremont_bridge,temperature_2m,precipitation,wind_speed_10m
0,2012-10-02,1938.0,13.300000,0.0,5.654545
1,2012-10-03,3521.0,11.495833,0.0,12.845833
2,2012-10-04,3475.0,11.750000,0.0,11.070833
3,2012-10-05,3148.0,12.079167,0.0,9.095833
4,2012-10-06,2006.0,12.454167,0.0,8.541667


In [39]:
daily_df.describe()

,date,fremont_bridge,temperature_2m,precipitation,wind_speed_10m
count,4990,4990.000000,4990.000000,4990.000000,4990.000000
mean,2019-08-01 12:00:00,2577.609619,11.180999,3.408818,10.828060
min,2012-10-02 00:00:00,11.000000,-7.554167,0.000000,2.537500
25%,2016-03-02 06:00:00,1611.250000,6.646875,0.000000,7.279167
50%,2019-08-01 12:00:00,2476.500000,10.581250,0.200000,9.535417
75%,2022-12-30 18:00:00,3439.750000,16.002083,4.200000,13.141667
max,2026-05-31 00:00:00,6428.000000,29.145833,77.000000,37.187500
std,NaN,1275.015367,5.793991,6.469149,5.006077


In [40]:
#Perform check of date span
(
    daily_df["date"].max()
    - daily_df["date"].min()
).days + 1

4990

In [43]:
#check for missing values
daily_df.isna().sum()

date              0
fremont_bridge    0
temperature_2m    0
precipitation     0
wind_speed_10m    0
dtype: int64

### Save Processed Dataset

The cleaned and aggregated daily dataset is written to
the `data/processed` directory. This dataset serves as
the primary input for the exploratory data analysis and
forecasting notebooks developed in subsequent phases of
the project.

In [42]:
#Write Out merged data
daily_df.to_csv(
    "../data/processed/bike_weather_daily.csv",
    index=False
)